# Text cleaning and component labels

Checks `nhtsa.text.clean_text` and `nhtsa.labels.component_labels` on the full
data, via `nhtsa.dataset.cleaned_complaints()`. Cleaning adds a `TEXT` column;
the original `CDESCR` is untouched.

In [1]:
import altair as alt
import polars as pl
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from nhtsa.dataset import cleaned_complaints
from nhtsa.labels import COMPONENT_LABELS, component_label
from nhtsa.notebook import show_texts, stats

pl.Config.set_tbl_rows(60).set_fmt_str_lengths(80);

In [2]:
# CHANGED: changed beyond what lowercasing and whitespace cleanup alone would do.
complaints = cleaned_complaints().with_columns(
    CHANGED=pl.col("TEXT")
    != pl.col("CDESCR")
    .str.to_lowercase()
    .str.replace_all(r"\s+", " ")
    .str.strip_chars()
)
chars_removed = complaints.select(
    1 - pl.col("TEXT").str.len_chars().sum() / pl.col("CDESCR").str.len_chars().sum()
).item()
stats(
    {
        "changed by cleaning": f"{complaints['CHANGED'].mean():.1%}",
        "characters removed": f"{chars_removed:.1%}",
        "hotline template": f"{complaints['HOTLINE'].mean():.1%}",
        "recall parts unavailable": f"{complaints['PARTS_UNAVAILABLE'].mean():.1%}",
        "empty after cleaning": f"{(complaints['TEXT'] == '').sum():,}",
    }
)

In [3]:
by_year = (
    complaints.group_by("YEAR")
    .agg(
        pl.col("CHANGED").mean().alias("changed by cleaning"),
        pl.col("HOTLINE").mean().alias("hotline template"),
        pl.col("PARTS_UNAVAILABLE").mean().alias("recall parts unavailable"),
    )
    .unpivot(index="YEAR", variable_name="flag", value_name="share")
)
alt.Chart(by_year).mark_line(point=True).encode(
    x=alt.X("YEAR:O", title=None),
    y=alt.Y("share:Q", axis=alt.Axis(format="%")),
    color="flag:N",
    tooltip=["YEAR", "flag", alt.Tooltip("share:Q", format=".1%")],
).properties(title="Share of complaints by year received", width=720)

alt.Chart(...)

## Before and after

Original `CDESCR` on the left, cleaned `TEXT` on the right. Change `SHOW` or
`SEED` and re-run the cell for a different sample.

In [4]:
FILTERS = {
    "changed by cleaning": pl.col("CHANGED"),
    "hotline template": pl.col("HOTLINE"),
    "recall parts unavailable": pl.col("PARTS_UNAVAILABLE"),
    "any": pl.lit(True),
}
SHOW = "changed by cleaning"  # a key of FILTERS
SEED = 0

for r in complaints.filter(FILTERS[SHOW]).sample(4, seed=SEED).iter_rows(named=True):
    display(
        show_texts(
            f"{r['ODINO']} · {r['YEAR']} · {', '.join(r['LABELS'])}",
            r["CDESCR"],
            r["TEXT"],
        )
    )

## What's left

Leftover `*` tokens and the most common openings, for spotting boilerplate the
patterns miss. "the contact stated…" is kept on purpose: the rest of a hotline
complaint describes the actual problem.

In [5]:
stars = (
    complaints.select(pl.col("TEXT").str.extract_all(r"\*\S{1,8}").alias("token"))
    .explode("token")
    .drop_nulls()["token"]
    .value_counts(sort=True)
    .head(15)
)
openings = (
    complaints.select(
        pl.col("TEXT").str.extract(r"^(\S+ \S+ \S+ \S+ \S+)").alias("opening")
    )
    .drop_nulls()["opening"]
    .value_counts(sort=True)
    .head(15)
)
display(stars, openings)

token,count
str,u32
"""**""",1818
"""***""",956
"""***.""",441
"""*******""",243
"""****""",216
"""*********""",214
"""*the""",111
"""*akthe""",93
"""***transf""",87


opening,count
str,u32
"""the contact stated that while""",52270
"""the contact stated that the""",42735
"""the contact received notification of""",31297
"""the contact stated while driving""",21684
"""takata recall. the contact received""",13370
"""the contact received a recall""",3557
"""the contact stated that after""",3419
"""the contact stated that when""",3059
"""while driving various speeds, the""",2647


## Does the text still reveal its era?

Complaints from just before NHTSA's 2021 system change (2017–2020) vs just after
(2022–2025), 25k each. A TF-IDF + logistic regression model tries to tell them
apart. Some signal is real (new vehicle tech, new recalls), so accuracy won't hit
50%, but formatting artifacts should account for much less after cleaning. The
top words show what still separates the eras; expect model years, recalls of
the period (e.g. Takata), street addresses in older complaints (newer ones are
redacted), and shifts in the hotline's wording.

In [6]:
def era_rows(first_year, last_year, after):
    return (
        complaints.filter(
            pl.col("YEAR").is_between(first_year, last_year), pl.col("TEXT") != ""
        )
        .sample(25_000, seed=0)
        .with_columns(AFTER=pl.lit(after))
    )


era_sample = pl.concat([era_rows(2017, 2020, False), era_rows(2022, 2025, True)])


def fit_era_model(texts, lowercase):
    vectorizer = TfidfVectorizer(lowercase=lowercase, min_df=5, sublinear_tf=True)
    x = vectorizer.fit_transform(texts)
    x_train, x_test, y_train, y_test = train_test_split(
        x, era_sample["AFTER"].to_numpy(), test_size=0.25, random_state=0
    )
    model = LogisticRegression(max_iter=1000).fit(x_train, y_train)
    return model.score(x_test, y_test), vectorizer, model


raw_acc, _, _ = fit_era_model(era_sample["CDESCR"].to_list(), lowercase=False)
lower_acc, _, _ = fit_era_model(era_sample["CDESCR"].to_list(), lowercase=True)
clean_acc, era_vectorizer, era_model = fit_era_model(
    era_sample["TEXT"].to_list(), lowercase=False
)
pl.DataFrame(
    {
        "text": ["raw (case kept)", "raw, lowercased", "cleaned (TEXT)"],
        "era accuracy": [f"{a:.1%}" for a in (raw_acc, lower_acc, clean_acc)],
    }
)

text,era accuracy
str,str
"""raw (case kept)""","""99.2%"""
"""raw, lowercased""","""80.3%"""
"""cleaned (TEXT)""","""77.8%"""


In [7]:
weights = pl.DataFrame(
    {"word": era_vectorizer.get_feature_names_out(), "weight": era_model.coef_[0]}
).sort("weight")
top_weights = pl.concat(
    [
        weights.head(20).with_columns(era=pl.lit("2017–2020")),
        weights.tail(20).with_columns(era=pl.lit("2022–2025")),
    ]
)
alt.Chart(top_weights).mark_bar().encode(
    x=alt.X("weight:Q", title="weight toward 2022–2025"),
    y=alt.Y("word:N", sort="x", title=None),
    color=alt.Color("era:N", title="predicts"),
    tooltip=["word", alt.Tooltip("weight:Q", format=".2f")],
).properties(title="Words that most separate the eras (cleaned text)", width=480)

alt.Chart(...)

## Component labels

NHTSA's 54 top-level component groups mapped to consistent labels
(`nhtsa.labels.COMPONENT_LABELS`). Unmapped groups would show as null.

In [8]:
mapping = (
    complaints.select("ODINO", "COMPONENTS")
    .explode("COMPONENTS")
    .select(
        "ODINO",
        group=pl.col("COMPONENTS").str.split(":").list.first(),
        label=component_label(pl.col("COMPONENTS")),
    )
    .unique(["ODINO", "group"])
    .group_by("label", "group")
    .agg(complaints=pl.len())
    .sort("label", "complaints", descending=[False, True])
)
unmapped = mapping.filter(pl.col("label").is_null(), pl.col("group").is_not_null())
print(
    f"{len(set(COMPONENT_LABELS.values()))} labels from {len(COMPONENT_LABELS)} "
    f"groups; {unmapped.height} unmapped groups in the data."
)
mapping

25 labels from 54 groups; 0 unmapped groups in the data.


label,group,complaints
str,str,u32
"""air_bags""","""AIR BAGS""",154123
"""brakes""","""SERVICE BRAKES, HYDRAULIC""",95924
"""brakes""","""SERVICE BRAKES""",79472
"""brakes""","""PARKING BRAKE""",4825
"""brakes""","""SERVICE BRAKES, AIR""",4231
"""brakes""","""SERVICE BRAKES, ELECTRIC""",2639
"""brakes""","""SERVICE BRAKES, HYDRAULIC; AUTOHOLD BRAKE SYSTEM/BRAKE HOLD""",9
"""child_seat""","""CHILD SEAT""",13993
"""child_seat""","""Chest Clip, Buckle, Harness""",509


In [9]:
label_counts = complaints.select(pl.col("LABELS").explode())["LABELS"].value_counts()
labels_per_complaint = complaints.select(
    pl.col("LABELS").list.len().clip(upper_bound=4).alias("labels")
)["labels"].value_counts()
(
    alt.Chart(label_counts)
    .mark_bar()
    .encode(
        x=alt.X("count:Q", title="complaints"),
        y=alt.Y("LABELS:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("count:Q", format=",")],
    )
    .properties(title="Complaints per label", width=420)
    | alt.Chart(labels_per_complaint)
    .mark_bar()
    .encode(
        x=alt.X("labels:O", title="labels per complaint (4 = 4+)"),
        y=alt.Y("count:Q", title="complaints"),
        tooltip=[alt.Tooltip("count:Q", format=",")],
    )
    .properties(width=200)
)

alt.HConcatChart(...)